In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_DIR = Path("../data")
SAMPLE_PATH = DATA_DIR / "loan_sample_100m.csv"
df=pd.read_csv(SAMPLE_PATH,low_memory=False)
df.dtypes[df.dtypes=="object"].index.tolist()#dtypes为object是指字符串类型
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199999 entries, 0 to 199998
Columns: 145 entries, id to settlement_term
dtypes: float64(59), int64(51), object(35)
memory usage: 221.3+ MB


In [14]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# 分析用列（见 docs/dataset.md）：基础/结果、转化、风险、收益、用户分群
COLS_TO_KEEP = [
    "loan_status", "issue_d", "loan_amnt", "funded_amnt", "funded_amnt_inv",
    "term", "int_rate", "installment", "grade", "sub_grade", "verification_status",
    "purpose", "addr_state", "zip_code", "title",
    "dti", "annual_inc", "emp_length", "emp_title", "home_ownership",
    "delinq_2yrs", "mths_since_last_delinq", "pub_rec", "revol_bal", "revol_util",
    "open_acc", "total_acc", "inq_last_6mths", "inq_last_12m", "earliest_cr_line",
    "collections_12_mths_ex_med", "chargeoff_within_12_mths",
    "total_pymnt", "total_rec_int", "total_rec_prncp", "last_pymnt_d", "last_pymnt_amnt",
    "out_prncp", "out_prncp_inv",
    "avg_cur_bal", "bc_util", "all_util", "tot_cur_bal", "total_rev_hi_lim",
    "acc_now_delinq", "num_accts_ever_120_pd", "num_tl_90g_dpd_24m", "pub_rec_bankruptcies",
    "application_type",
]
existing = [c for c in COLS_TO_KEEP if c in df.columns]
df = df[existing].copy()
print("保留列数:", len(existing), "| 删除列数:", 145 - len(existing))
df.head(10)

保留列数: 49 | 删除列数: 96


,loan_status,issue_d,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,avg_cur_bal,bc_util,all_util,tot_cur_bal,total_rev_hi_lim,acc_now_delinq,num_accts_ever_120_pd,num_tl_90g_dpd_24m,pub_rec_bankruptcies,application_type
0,Current,Dec-2018,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,1878.0,5.9,28.0,16901,42000,0,0,0,1,Individual
1,Current,Dec-2018,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,24763.0,8.3,57.0,321915,50800,0,0,0,1,Individual
2,Current,Dec-2018,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,18383.0,0.0,35.0,110299,24100,0,0,0,0,Individual
3,Current,Dec-2018,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,30505.0,75.2,70.0,305049,7000,0,0,0,0,Individual
4,Current,Dec-2018,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,9667.0,8.9,54.0,116007,23100,0,0,0,0,Individual
5,Current,Dec-2018,5550,5550,5550.0,36 months,15.02,192.45,C,C3,...,40338.0,64.0,58.0,685749,111900,0,0,0,0,Individual
6,Current,Dec-2018,2000,2000,2000.0,36 months,17.97,72.28,D,D1,...,854.0,NaN,100.0,854,0,0,0,0,0,Individual
7,Current,Dec-2018,6000,6000,6000.0,36 months,13.56,203.79,C,C1,...,5085.0,90.8,74.0,91535,55500,0,0,0,0,Individual
8,Current,Dec-2018,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,5235.0,35.9,73.0,41882,22800,0,2,0,0,Individual
9,Current,Dec-2018,6000,6000,6000.0,36 months,14.47,206.44,C,C2,...,9197.0,60.6,48.0,349502,132500,0,0,0,0,Individual


In [49]:
null_column_count=df.isnull().sum() #计算每列的缺失值数量
null_column_pct=(null_column_count/len(df)).round(4)
missing_df=pd.DataFrame({'n_null':null_column_count,'null_pct':null_column_pct})#把null_column_count和pct做成dataframe表格
missing_df=missing_df.sort_values('n_null',ascending=False)
missing_df.head(20)

,n_null,null_pct
mths_since_last_delinq,112455,0.5623
emp_title,29627,0.1481
emp_length,17852,0.0893
bc_util,2704,0.0135
dti,392,0.0020
last_pymnt_d,241,0.0012
revol_util,231,0.0012
all_util,56,0.0003
avg_cur_bal,21,0.0001
earliest_cr_line,0,0.0000


In [52]:
#mths_since_last_delinq（上次逾期距今月数） 是非常重要的预测指标，但这里数据缺失较严重，考虑合理填补数据
print(f'现有数据中，距上次逾期月数最大是{df.mths_since_last_delinq.max()}个月')
#上面显示，距上次逾期月数最大是179.0个月，所以我们把mths_since_last_delinq为null的值设为180，表示该客户上次逾期刚好在180个月前
#当然，实际上他可能一直都没逾期或者上次逾期月数大于180，这里难免与实际情况有一定差距
n_fill=df['mths_since_last_delinq'].isna().sum()
print(f'要补全{n_fill}个mths_since_last_delinq空值')

现有数据中，距上次逾期月数最大是179.0个月
要补全112455个mths_since_last_delinq空值


In [55]:
df['mths_since_last_delinq']=df['mths_since_last_delinq'].fillna(180)
df['mths_since_last_delinq'].describe()

count    199999.000000
mean        117.372307
std          72.436955
min           0.000000
25%          39.000000
50%         180.000000
75%         180.000000
max         180.000000
Name: mths_since_last_delinq, dtype: float64

In [68]:
#emp_title和emp_length都比较难直接补全，其它指标缺失比例都比较小，故把剩下有空白值的行直接删除
before=df.shape[0]
df=df.dropna()
print(f'原行数为{before}，现行数为{df.shape[0]},删除了{before-df.shape[0]}行')

原行数为199999，现行数为167981,删除了32018行


In [70]:
df['loan_status'].value_counts()

loan_status
Current               160003
Fully Paid              6003
Late (31-120 days)       881
In Grace Period          744
Late (16-30 days)        251
Charged Off               99
Name: count, dtype: int64

### 优质客户和一般客户的定义（is_default）
根据`loan_status` 定义客户类型，Current和Fully Paid 定义为优质客户(0)，其他为一般客户(1)，一般客户按逾期程度依次为In Grace Period、Late (16-30 days)、Late (31-120 days)和Charged Off

In [ ]:
default_status=['Late (31-120 days)','In Grace Period','Late (16-30 days)','Charged Off']
df['is_default']=df['loan_status'].isin(default_status).astype(int)
df['is_default'].value_counts()

/var/folders/z_/s02m_tcd6x750hc3d4jq5njm0000gp/T/ipykernel_486/2858283813.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['is_default']=df['loan_status'].isin(default_status).astype(int)


is_default
0    166006
1      1975
Name: count, dtype: int64

In [79]:
df['int_rate'].describe()

count    167981.000000
mean         12.856414
std           5.167734
min           6.000000
25%           8.460000
50%          11.800000
75%          16.140000
max          30.990000
Name: int_rate, dtype: float64

In [83]:
df['int_rate'].value_counts()

int_rate
13.56    9078
6.11     9005
15.02    8623
14.47    7933
7.84     7544
11.55    7422
16.14    7373
16.91    7313
8.46     6572
10.47    6374
12.73    5906
7.21     5876
10.08    5694
17.97    5562
6.67     5472
18.94    5188
11.06    4981
19.92    4716
8.19     4681
11.80    4177
20.89    3894
8.81     3483
6.46     3334
10.72    3292
12.98    3253
22.35    3071
10.33    2856
11.31    2445
7.56     2400
7.02     2331
23.40    1938
27.27    1654
25.34    1252
24.37    1138
26.31    1092
28.72     354
29.69     195
30.17     127
30.79     104
30.65      95
30.75      84
30.84      34
6.00       20
30.89      18
30.99      14
30.94      11
13.06       1
6.19        1
Name: count, dtype: int64

In [ ]:
 #提取月份数，并在后面把它转换为数字
df['term'].value_counts()
df['term_months']=df['term'].str[1:3] 
print(df['term'].value_counts())
print(df['term_months'].value_counts())
df['term_months'].dtype


term
36 months    113779
60 months     54202
Name: count, dtype: int64
term_months
36    113779
60     54202
Name: count, dtype: int64


/var/folders/z_/s02m_tcd6x750hc3d4jq5njm0000gp/T/ipykernel_486/2158106811.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['term_months']=df['term'].str[1:3]


dtype('O')

In [ ]:
df['term_months']=pd.to_numeric(df['term_months']) #把月份数转化为数字类型
print(df['term_months'].value_counts())
df['term_months'].dtype

term_months
36    113779
60     54202
Name: count, dtype: int64


/var/folders/z_/s02m_tcd6x750hc3d4jq5njm0000gp/T/ipykernel_486/3129166263.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['term_months']=pd.to_numeric(df['term_months'])


dtype('int64')

In [ ]:
#把issue_d由月份转到日期格式，每月的日期均统一假定为1日
print(df['issue_d'].value_counts())
df['issue_d'].dtype
df['issue_date']=pd.to_datetime(df['issue_d'],format='%b-%Y')
print(df['issue_date'].value_counts())


issue_d
Oct-2018    38403
Nov-2018    34630
Sep-2018    33650
Dec-2018    32901
Aug-2018    28397
Name: count, dtype: int64
issue_date
2018-10-01    38403
2018-11-01    34630
2018-09-01    33650
2018-12-01    32901
2018-08-01    28397
Name: count, dtype: int64


/var/folders/z_/s02m_tcd6x750hc3d4jq5njm0000gp/T/ipykernel_486/3768868827.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['issue_date']=pd.to_datetime(df['issue_d'],format='%b-%Y')


### 清洗小结

- 上面做了一些清洗步骤，主要做了填补空白数据、`is_default` 定义、`term`/`issue_d` 的格式处理。

In [ ]:
#看看清洗完以后的数据
df.describe(include='all') # include='all'是为了要把所有列都列出来，否则只对数值列做统计

,loan_status,issue_d,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,tot_cur_bal,total_rev_hi_lim,acc_now_delinq,num_accts_ever_120_pd,num_tl_90g_dpd_24m,pub_rec_bankruptcies,application_type,is_default,term_months,issue_date
count,167981,167981,167981.000000,167981.000000,167981.000000,167981,167981.000000,167981.000000,167981,167981,...,1.679810e+05,1.679810e+05,167981.0,167981.000000,167981.000000,167981.000000,167981,167981.000000,167981.000000,167981
unique,6,5,NaN,NaN,NaN,2,NaN,NaN,7,35,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
top,Current,Oct-2018,NaN,NaN,NaN,36 months,NaN,NaN,A,A1,...,NaN,NaN,NaN,NaN,NaN,NaN,Individual,NaN,NaN,NaN
freq,160003,38403,NaN,NaN,NaN,113779,NaN,NaN,50701,12339,...,NaN,NaN,NaN,NaN,NaN,NaN,147076,NaN,NaN,NaN
mean,NaN,NaN,16321.180818,16321.180818,16318.340089,NaN,12.856414,471.941622,NaN,NaN,...,1.526429e+05,3.999457e+04,0.0,0.464154,0.059102,0.116531,NaN,0.011757,43.744019,2018-10-03 00:24:06.335002112
min,NaN,NaN,1000.000000,1000.000000,725.000000,NaN,6.000000,30.480000,NaN,NaN,...,0.000000e+00,2.000000e+02,0.0,0.000000,0.000000,0.000000,NaN,0.000000,36.000000,2018-08-01 00:00:00
25%,NaN,NaN,9000.000000,9000.000000,9000.000000,NaN,8.460000,261.270000,NaN,NaN,...,3.030800e+04,1.770000e+04,0.0,0.000000,0.000000,0.000000,NaN,0.000000,36.000000,2018-09-01 00:00:00
50%,NaN,NaN,14500.000000,14500.000000,14500.000000,NaN,11.800000,395.160000,NaN,NaN,...,8.387100e+04,3.060000e+04,0.0,0.000000,0.000000,0.000000,NaN,0.000000,36.000000,2018-10-01 00:00:00
75%,NaN,NaN,22400.000000,22400.000000,22400.000000,NaN,16.140000,634.080000,NaN,NaN,...,2.312480e+05,5.070000e+04,0.0,0.000000,0.000000,0.000000,NaN,0.000000,60.000000,2018-11-01 00:00:00
max,NaN,NaN,40000.000000,40000.000000,40000.000000,NaN,30.990000,1618.240000,NaN,NaN,...,9.971659e+06,1.514001e+06,0.0,58.000000,58.000000,6.000000,NaN,1.000000,60.000000,2018-12-01 00:00:00


In [114]:
#导出清洗后数据，方便使用
df.to_csv(DATA_DIR / "myself_loan_clean.csv", index=False)